# Week 44, Day 1: Learning from surprise: the Rescorla-Wagner model

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w044/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Acquisition and extinction

Write rw(trials, alpha): trials is a list of (x, r) pairs, where x is a vector of which cues are present. Compute the prediction error, update V, and return the final V and the history of V after each trial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rw(trials, alpha=0.2, n_cues=2):
    V = np.zeros(n_cues); hist = []
    for x, r in trials:
        x = np.asarray(x, float)
        ...
        hist.append(V.copy())
    return V, np.array(hist)

light = [1, 0]
V, hist = rw([(light, 1)] * 50 + [(light, 0)] * 50)
print(f"after acquisition V_light = {hist[49, 0]:.3f}; after extinction V_light = {hist[-1, 0]:.4f}")
plt.plot(hist[:, 0]); plt.axvline(50, color="k", ls=":"); plt.xlabel("trial"); plt.ylabel("V (light)"); plt.show()

In [ ]:
import numpy as np
def test_acquire():
    "after 50 rewarded trials the light's strength is above 0.99"
    assert hist[49, 0] > 0.99
def test_extinguish():
    "after 50 unrewarded trials it is below 0.01"
    assert hist[-1, 0] < 0.01
def test_curve():
    "the first trial raises V by exactly alpha"
    assert abs(hist[0, 0] - 0.2) < 1e-9

test_acquire()
test_extinguish()
test_curve()
print("All checks passed")

### Exercise: Competition for prediction

Simulate three designs with rw: blocking (light alone rewarded 50 trials, then light plus tone rewarded 50), the control (light plus tone rewarded 50 trials), and overexpectation (light alone rewarded 50, tone alone rewarded 50, then both together rewarded 50).

In [ ]:
tone, both = [0, 1], [1, 1]
V_block, _ = ...
V_ctrl, _ = ...
V_over, h_over = ...
print(f"blocking: light {V_block[0]:.3f}, tone {V_block[1]:.4f}")
print(f"control:  light {V_ctrl[0]:.3f}, tone {V_ctrl[1]:.3f}")
print(f"overexpectation: light {V_over[0]:.3f}, tone {V_over[1]:.3f}")

In [ ]:
import numpy as np
def test_block():
    "after pretraining on the light, the tone gains less than 0.01"
    assert V_block[1] < 0.01
def test_control():
    "without pretraining, light and tone share the prediction equally"
    assert abs(V_ctrl[0] - 0.5) < 0.01 and abs(V_ctrl[1] - 0.5) < 0.01
def test_over():
    "overexpectation: two cues each trained to 1, then trained together to 1, both fall below 0.6"
    assert V_over[0] < 0.6 and V_over[1] < 0.6

test_block()
test_control()
test_over()
print("All checks passed")

### Exercise: What the model misses

Compare learning about the light after 50 pre-exposure trials (light alone, no reward) with learning from scratch: the 50 rewarded trials should give identical curves in the model, which is the prediction latent inhibition refutes.

In [ ]:
_, h_pre = ...
_, h_fresh = ...
print("largest difference between the two learning curves:", np.abs(h_pre[50:] - h_fresh).max())

In [ ]:
import numpy as np
def test_no_latent():
    "in Rescorla-Wagner, 50 unrewarded pre-exposures leave learning unchanged"
    assert np.allclose(h_pre[50:], h_fresh)

test_no_latent()
print("All checks passed")